Complete Preprocessing Pipeline with ColumnTransformer


Production-ready preprocessing pipeline handling mixed data types with automatic leakage prevention


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import RobustScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load # Import dataset_load


# FIX: Specify the correct file name for the dataset
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version using dataset_load to avoid deprecation warning
df = dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

# FIX: Print data loaded message
print(f"Data Loaded from Kaggle dataset 'blastchar/telco-customer-churn' file: {file_path}")

# FIX: Handle 'TotalCharges' column which might be object type due to spaces/empty strings
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Uncomment for data inspection
df.describe()

# FIX: Update target column for the new dataset
target = 'Churn'
y = df[target]

# 'X' is everything ELSE (drop the target column)
# We also drop IDs and dates for now, as standard models can't process them directly
# FIX: Update columns to drop for the new dataset
columns_to_drop = [target,'customerID']
X = df.drop(columns=columns_to_drop, errors='ignore')


# ============================================================
# STEP 1: Identify column types
# ============================================================
# In practice, inspect with df.info() and df.dtypes

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
# ------------------------------------------------------------

# ============================================================
# STEP 2: Build sub-pipelines for each type
# ============================================================

# impute missings with median
# numeric_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler())
# ])
# we have outliers so we should use RobustScaler()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

#impute missing with 'Unknown', then one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',fill_value='Unknown')),
    ('scaler', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

# ============================================================
# STEP 3: Combine with ColumnTransformer
# ============================================================
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder = 'drop' # drop unlisted columns
)

# ============================================================
# STEP 4: Full pipeline = preprocessing + model
# ============================================================

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# ============================================================
# STEP 5: Split FIRST, then fit (leakage-free)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
full_pipeline.fit(X_train,y_train) # transformation + model
score = full_pipeline.score(X_test,y_test) # transformation + prediction

# Cross-validation is also leak-free with Pipeline:
scores = cross_val_score(full_pipeline, X, y, cv=5)
print(f"CV Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

print("Pipeline steps: ", [name for name, _ in full_pipeline.steps])
print("Preprocessor transformers: ", [name for name,_,_  in preprocessor.transformers])

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Data Loaded from Kaggle dataset 'blastchar/telco-customer-churn' file: WA_Fn-UseC_-Telco-Customer-Churn.csv
CV Accuracy: 0.747 +/- 0.006
Pipeline steps:  ['preprocess', 'model']
Preprocessor transformers:  ['num', 'cat']


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict on the test set
y_pred = full_pipeline.predict(X_test)

print("\n" + "=" * 60)
print("THE REALITY OF ACCURACY")
print("=" * 60)

print("\nConfusion Matrix:")
# This shows [True Negatives, False Positives]
#            [False Negatives, True Positives]
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
# This shows Precision, Recall, and F1-Score
print(classification_report(y_test, y_pred, zero_division=0))


THE REALITY OF ACCURACY

Confusion Matrix:
[[743 295]
 [ 69 302]]

Classification Report:
              precision    recall  f1-score   support

          No       0.92      0.72      0.80      1038
         Yes       0.51      0.81      0.62       371

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.81      0.74      0.76      1409



Imputation Strategy Selection

Detecting missingness type and choosing the right imputation approach




In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer

# ============================================================
# DETECT MISSINGNESS PATTERN
# ============================================================

def diagnose_missingness(df):
  "Analyze missing data patterns to choose imputation strategy."
  missing_pct = df.isnull().mean() * 100
  print(f"Missing value percentage: {missing_pct[missing_pct > 0].sort_values(ascending=False)}")

  missing_indicators = df.isnull().astype(int)
  for col in df.columns[df.isnull().any()]:
    corr_with_others = missing_indicators[col].corr(
        df.drop(columns=[col]).select_dtypes(include=np.number).mean(axis=1)
    )
    pattern = "MCAR" if abs(corr_with_others) < 0.1 else "MAR/MNAR"
    print(f" {col}: correlation = {corr_with_others:.3f} -> likely {pattern}")

  return missing_pct


# ============================================================
# APPLY STRATEGY BASED ON DIAGNOSIS
# ============================================================
def smart_impute(df, col, strategy='auto'):
    """Impute a column based on missingness pattern.
    WARNING: In production, fit imputers on train set only,
    then transform test set separately to avoid data leakage.
    """

    if(strategy == 'auto'):
      missing_rate = df[col].isnull().mean()
      if(missing_rate < 0.05):
        strategy = 'median'  # Low missing: simple imputation
      elif missing_rate > 0.4:
        strategy = 'indicator' # High missing signal in the gap
      else:
        strategy = 'knn'


    if(strategy == 'median'):
      imputer = SimpleImputer(strategy='median')
      df[col] = imputer.fit_transform(df[col].values.reshape(-1, 1))

    elif strategy == 'knn':
      # KNN uses similar rows to estimate missing values
      imputer = KNNImputer(n_neighbors=5)
      numeric_cols = df.select_dtypes(include=np.number).columns
      df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    elif strategy == 'indicator':
      # MNAR: the gap IS the signal - preserve it
      df[f'{col}_missing'] = df[col].isnull().astype(int)
      imputer = SimpleImputer(strategy='median')
      df[col] = imputer.fit_transform(df[col].values.reshape(-1, 1))

    return df

df = dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

# FIX: Ensure TotalCharges is numeric and handle potential NaNs from coercion
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df.info()

diagnose_missingness(df)
# Test your smart imputer on the TotalCharges column
print("Before imputation:", df['TotalCharges'].isnull().sum(), "missing values")

df = smart_impute(df, 'TotalCharges', strategy='auto')

print("After imputation:", df['TotalCharges'].isnull().sum(), "missing values")



Using Colab cache for faster access to the 'telco-customer-churn' dataset.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract      